In [ ]:
# ================================
# Customer Segmentation - Task 2
# ================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Load Dataset
df = pd.read_csv('/content/online_retail.csv', encoding='ISO-8859-1')

# Display Dataset
print(df.head())

# Remove Missing Customer IDs
df = df.dropna(subset=['CustomerID'])

# Remove Duplicates
df = df.drop_duplicates()

# Create Total Price
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Customer-wise Data
customer_data = df.groupby('CustomerID').agg({
    'Quantity':'sum',
    'TotalPrice':'sum'
}).reset_index()

# Feature Selection
X = customer_data[['Quantity','TotalPrice']]

# Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow Method
wcss = []

for i in range(1,11):
    model = KMeans(n_clusters=i, random_state=42, n_init=10)
    model.fit(X_scaled)
    wcss.append(model.inertia_)

plt.figure(figsize=(6,4))
plt.plot(range(1,11), wcss, marker='o')
plt.title("Elbow Method")
plt.xlabel("Clusters")
plt.ylabel("WCSS")
plt.show()

# KMeans
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_data['Cluster'] = kmeans.fit_predict(X_scaled)

# Scatter Plot
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=customer_data,
    x='Quantity',
    y='TotalPrice',
    hue='Cluster',
    palette='Set2'
)
plt.title("Customer Segmentation")
plt.show()

# Cluster Summary
print(customer_data.groupby('Cluster').mean())

# Save Output
customer_data.to_csv("Customer_Segmentation_Output.csv", index=False)

print("Task 2 Completed Successfully!")